# League of Legends Match History 
**Author**: maxk678 | **Data Source:** [Riot API](https://developer.riotgames.com/)

This notebook demonstrates an end-to-end data pipeline for a League of Legends player
1. **Data Collection**
2. **Data Enrichment**
3. **Storage**

> ⚠️ **Setup:** Requires a `.env` file with `riot_api_key`, `db_username`, `db_password`, `db_host`, and `db_port`.

---
## 1. Imports & API Setup

Load environment variables and define all API helper functions for the Riot Games API.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import requests
import pandas as pd
import time

api_key = os.environ.get('riot_api_key')


def get_puuid(gameName=None, tagLine=None, region='americas'):
    """Gets the puuid from riot_id and riot_tag

    Args:
        gameName (str,optional): Riot ID. Defaults to None.
        tagLine (str,optional): Riot Tag. Defaults to None.
        region (str,optional): Region. Defaults to 'americas'.
    Returns:
        str:puuid
    """
    root_url= f'https://{region}.api.riotgames.com/'
    endpoint= f'riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}'

    response = requests.get(root_url+endpoint+'?api_key='+api_key)

    return response.json()['puuid']

def get_id_tag_from_puuid(puuid=None):
    """Gets the riot_id and riot_tag from a puuid

    Args:
        puuid (str, optional): puuid, Defaults to None.

    Returns:
        id (dict): Dictionary with riot_id and riot_tag
    """

    root_url = 'https://americas.api.riotgames.com/'
    endpoint = 'riot/account/v1/accounts/by-puuid/'

    response = requests.get(root_url + endpoint + puuid + '?api_key=' + api_key)

    id = {
        'gameName': response.json()['gameName'],
        'tagLine' : response.json()['tagLine']
    }

    return id

def get_ladder(top=10):
    """Gets the top x players in soloq

    Args:
        top (int, optional): Number of players to return. Defaults to 300.
    Returns:
        DataFrame: Returns a DataFrame of the top x players in soloq
    """

    root_url = 'https://na1.api.riotgames.com/'
    challenger = 'lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5'
    grandmaster = 'lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5'
    master = 'lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'

    chall_resp = requests.get(root_url + challenger + '?api_key=' + api_key)
    chall_df = pd.DataFrame(chall_resp.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

    gm_df = pd.DataFrame()
    m_df = pd.DataFrame()
    if top > 300:
            gm_resp = requests.get(root_url + grandmaster + '?api_key=' + api_key)
            gm_df = pd.DataFrame(gm_resp.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    if top > 1000:
            m_resp = requests.get(root_url + master + '?api_key=' + api_key)
            m_df = pd.DataFrame(m_resp.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

    df = pd.concat([chall_df, gm_df, m_df]).reset_index(drop=True)[:top]

    df = df.reset_index().drop(columns=['rank']).rename(columns={'index':'rank'})
    df['rank'] += 1

    return df  

def get_match_history(region=None, puuid=None, start_time=None, end_time=None, queue=None, count=None):
    """Gets match IDs for a given player.

    Args:
        region (str): Regional endpoint (e.g. 'americas').
        puuid (str): Player UUID.
        start_time (int, optional): Start of date range as Unix timestamp. Defaults to None.
        end_time (int, optional): End of date range as Unix timestamp. Defaults to None.
        queue (int, optional): Queue type filter (e.g. 420 for ranked SoloQ). Defaults to None.
        count (int, optional): Max number of match IDs to return. Defaults to None (fetches all).

    Returns:
        list: List of match ID strings.
    """
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/by-puuid/{puuid}/ids'
    
    all_match_ids = []
    start_index = 0
    batch_size = 100

    while True:
        # If count is set, don't fetch more than needed
        remaining = count - len(all_match_ids) if count else batch_size
        if remaining <= 0:
            break
        fetch_size = min(batch_size, remaining)

        query_params = f'?start={start_index}&count={fetch_size}'
        if start_time:
            query_params += f'&startTime={start_time}'
        if end_time:
            query_params += f'&endTime={end_time}'
        if queue:
            query_params += f'&queue={queue}'

        response = requests.get(root_url + endpoint + query_params + '&api_key=' + api_key)
        batch = response.json()

        if not batch:
            break

        all_match_ids.extend(batch)
        start_index += fetch_size
        time.sleep(2)

    return all_match_ids

def get_match_data_from_id(region=None, matchId=None):
    """Gets match data for a given match ID.

    Args:
        region (str): Regional endpoint (e.g. 'americas').
        matchId (str): Match ID string (e.g. 'NA1_5490303087').

    Returns:
        dict: Raw match JSON from the Riot API.
    """
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'/lol/match/v5/matches/{matchId}'    

    response = requests.get(root_url + endpoint + '?api_key=' + api_key)

    return response.json()

def process_match_json(match_json, puuid):
    """Processes match data

    Args:
        match_json (dict): Raw match JSON returned by get_match_data_from_id.
        puuid (str): Player UUID used to identify the target player within the match.

    Returns:
        DataFrame: Single-row DataFrame containing the player's match stats.
    """
    ## Architecture
    metadata = match_json['metadata']
    info = match_json['info']
    players = info['participants']
    participants = metadata['participants']
    teams = info['teams']
    player = players[participants.index(puuid)]
    perks = player['perks']
    stats = perks['statPerks']
    styles = perks['styles']

    primary = styles[0]
    secondary = styles[1]

    side_dict = {
        100:'blue',
        200: 'red'
    }

    ## Information
    match_id = metadata['matchId']


    game_creation = info['gameCreation']
    game_duration = info['gameDuration']
    game_start_timestamp = info['gameStartTimestamp']
    game_end_timestamp = info['gameEndTimestamp']
    patch = info['gameVersion']
    game_mode = info['gameMode']

    riot_id = player['riotIdGameName']
    riot_tag = player['riotIdTagline']
    summoner_id = player['summonerId']
    summoner_name = player['summonerName']

    
    win = player['win']

    champ_id = player['championId']
    champ_level = player['champLevel']

    gold_earned = player['goldEarned']
    neutral_minions_killed = player['neutralMinionsKilled']
    total_minions_killed = player['totalMinionsKilled']

    kills = player['kills']
    deaths = player['deaths']
    assists = player['assists']
    first_blood = player['firstBloodKill']

    total_damage_dealt = player['totalDamageDealtToChampions']
    total_damage_shielded = player['totalDamageShieldedOnTeammates']
    total_damage_taken = player['totalDamageTaken']
    total_damage_healed = player['totalHealsOnTeammates']
    total_time_cc_dealt = player['totalTimeCCDealt']

    early_surrender = player['gameEndedInEarlySurrender']
    surrender = player['gameEndedInSurrender']

    item0 = player['item0']
    item1 = player['item1']
    item2 = player['item2']
    item3 = player['item3']
    item4 = player['item4']
    item5 = player['item5']
    item6 = player['item6']

    summoner_1_id = player['summoner1Id']
    summoner_2_id = player['summoner2Id']



    defense = stats['defense']
    flex = stats['flex']
    offense = stats['offense']

    primary_style = primary['style']
    secondary_style = primary['style']

    primary_keystone = primary['selections'][0]['perk']
    primary_perk_1 = primary['selections'][1]['perk']
    primary_perk_2 = primary['selections'][2]['perk']
    primary_perk_3 = primary['selections'][3]['perk']

    secondary_perk_1 = secondary['selections'][0]['perk']
    secondary_perk_2 = secondary['selections'][1]['perk']

    objectives_stolen = player['objectivesStolen']
    objectives_stolen_assists = player['objectivesStolenAssists']


    matchDF = pd.DataFrame({
        'match_id' : [match_id],
        'participants' :[participants],
        'game_creation' : [game_creation],
        'game_duration' : [game_duration],
        'game_start_timestamp' : [game_start_timestamp],
        'game_end_timestamp': [game_end_timestamp],
        'game_mode' : [game_mode],
        'patch' : [patch],
        'puuid' : [puuid],
        'riot_id' : [riot_id],
        'riot_tag' : [riot_tag],
        'summoner_id' : [summoner_id],
        'summoner_name' : [summoner_name],
        'win' : [win],
        'champion' : [champ_id],
        'champion_level' : [champ_level],
        'kills' : [kills],
        'deaths' : [deaths],
        'assists' : [assists],
        'summoner_1_id' : [summoner_1_id],
        'summoner_2_id' : [summoner_2_id],
        'gold_earned' : [gold_earned],
        'total_minions_killed' : [total_minions_killed],
        'total_neutral_minions_killed' : [neutral_minions_killed],
        'early_surrender' : [early_surrender],
        'surrender' : [surrender],
        'first_blood' : [first_blood],
        'objectives_stolen' : [objectives_stolen],
        'objectives_stolen_assists' : [objectives_stolen_assists],
        'total_damage_dealt_champions' : [total_damage_dealt],
        'total_damage_taken' : [total_damage_taken],
        'total_damage_shielded_teammates' : [total_damage_shielded],
        'total_heals_teammates' : [total_damage_healed],
        'total_time_crowd_controlled' : [total_time_cc_dealt],
        'item0' : [item0],
        'item1' : [item1],
        'item2' : [item2],
        'item3' : [item3],
        'item4' : [item4],
        'item5' : [item5],
        'item6' : [item6],
        'perk_keystone' : [primary_keystone],
        'perk_primary_row_1' : [primary_perk_1],
        'perk_primary_row_2' : [primary_perk_2],
        'perk_primary_row_3' : [primary_perk_3],
        'perk_secondary_row_1' : [secondary_perk_1],
        'perk_secondary_row_2' :[secondary_perk_2],
        'perk_primary_style' : [primary_style],
        'perk_secondary_style' : [secondary_style],
        'perk_shard_defense' : [defense],
        'perk_shard_flex' : [flex],
        'perk_shard_offense' : [offense],
    })

    return matchDF

---
# 2. Player Lookup

Verify the API connection by looking up the target player's PUUID (a unique persistent player identifier) and reversing the lookup to confirm the round-trip works correctly

In [2]:
ladder = get_ladder(top=300)
ladder[['rank', 'puuid', 'leaguePoints', 'wins', 'losses']]

,rank,puuid,leaguePoints,wins,losses
0,1,OKkwOLkb0X0kDK8HmroYpxmSB8Yvn9GojxHHhQZfJRsQUU...,3460,188,87
1,2,ih9_0UsD72F5_2ZsSTGo67NdkEPsT24ReE8ixxM7INxXaN...,3277,345,227
2,3,PmsquLZOxsOZ65E5wL8XpzPmF6NvPZfcub6b5R-ujmxeoJ...,3225,241,184
3,4,sPSCdjYElrp7CmztZUzJPSSwpB0tzWV3rzrVpg3XxeHNQv...,3006,236,145
4,5,UjhzdUaIYqKx_bQLzlKkT6i4slo3o2OspAxlp_-81FQAX4...,2710,206,136
...,...,...,...,...,...
295,296,aYWUOc0aaz7_KfeY8xi2tvdRljbnAvpiuOFTSGBzmLijDe...,1405,187,149
296,297,BGVsD2ylFjHMIhHk4NBZw666cGukxYlavrV1j6BBu2ceVU...,1405,97,69
297,298,zMI-AbV7LoENpcs6zQs5EAKxA7CsiIgLZZkREveXCxvUbr...,1403,237,195
298,299,eF_1724_cv_B1VC_9RXjZ6i9vxks6OqH0a_NLySXBU1z2m...,1395,164,105


In [6]:
# Option A — pick a player from the ladder above by index (0 = rank 1)
#puuid = ladder['puuid'].iloc[150]

# Option B - look up a specific player by Riot ID
GAME_NAME = 'TaricMidGG'
TAG_LINE = 'NA1'
REGION = 'americas'
puuid = get_puuid(gameName=GAME_NAME, tagLine=TAG_LINE, region=REGION)

#Prints the PUUID of the option you picked
print(f'PUUID: {puuid}')

PUUID: t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw


In [7]:
# Reverse lookup - confirm PUUID maps back to the correct Riot ID
get_id_tag_from_puuid(puuid=puuid)

{'gameName': 'TaricMidGG', 'tagLine': 'NA1'}

---
# 3. Fetch Match History

Retrieve the 50 most recent match IDs for the player, then loop through each to fetch mutch match data and extract per-game stats into a single DataFrame.

In [9]:
# Fetch most recent match IDs
# --- OPTION A: Pull a full season ---
#start_date = int(time.mktime(time.strptime("2025-01-01", "%Y-%m-%d")))
#end_date   = int(time.mktime(time.strptime("2026-01-01", "%Y-%m-%d")))
#match_ids = get_match_history(region=REGION, puuid=puuid, start_time=start_date, end_time=end_date)

# --- OPTION B: Pull N most recent ---
match_ids = get_match_history(region=REGION, puuid=puuid, count=100)

print(f"Fetched {len(match_ids)} match IDs")
match_ids

Fetched 100 match IDs


['NA1_5490303087',
 'NA1_5487109265',
 'NA1_5478122035',
 'NA1_5477493562',
 'NA1_5477456755',
 'NA1_5477429141',
 'NA1_5477409582',
 'NA1_5477380233',
 'NA1_5477369297',
 'NA1_5476798721',
 'NA1_5476773080',
 'NA1_5475838808',
 'NA1_5475820811',
 'NA1_5475745537',
 'NA1_5475715148',
 'NA1_5475693275',
 'NA1_5475608898',
 'NA1_5475585467',
 'NA1_5475565655',
 'NA1_5475554304',
 'NA1_5474635852',
 'NA1_5474597505',
 'NA1_5474579126',
 'NA1_5474555845',
 'NA1_5473614870',
 'NA1_5473559917',
 'NA1_5472583721',
 'NA1_5472551887',
 'NA1_5472532432',
 'NA1_5466228278',
 'NA1_5466177447',
 'NA1_5466150656',
 'NA1_5466139268',
 'NA1_5464559933',
 'NA1_5464537351',
 'NA1_5463756319',
 'NA1_5463733904',
 'NA1_5463685369',
 'NA1_5462887940',
 'NA1_5462871272',
 'NA1_5462850334',
 'NA1_5459748889',
 'NA1_5459722557',
 'NA1_5452276643',
 'NA1_5452269445',
 'NA1_5446061847',
 'NA1_5445564962',
 'NA1_5445545750',
 'NA1_5444703870',
 'NA1_5443910460',
 'NA1_5443890079',
 'NA1_5443871266',
 'NA1_544356

In [10]:
# Loop through match IDs and build the DataFrame
df = pd.DataFrame()
errors = []

for match_id in match_ids:
    while True:
        try:
            game = get_match_data_from_id(region=REGION, matchId=match_id)
            if 'status' in game and game['status']['status_code'] == 429:
                print(f"Rate limited, waiting 30 seconds...")
                time.sleep(30)
                continue
            break
        except Exception as e:
            print(f"Connection error, waiting 30 seconds... ({e})")
            time.sleep(30)
            continue

    if 'metadata' not in game:
        print(f"Skipping {match_id}: {game}")
        errors.append(match_id)
        continue

    matchDF = process_match_json(game, puuid=puuid)
    df = pd.concat([df, matchDF])
    time.sleep(2)

df = df.reset_index(drop=True)
print(f"Shape: {df.shape}")
print(f"Skipped {len(errors)} matches")
df.head()

Shape: (100, 52)
Skipped 0 matches


,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,...,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,NA1_5490303087,[oXOnGvsfHrMrBDxweAq6tU7MB5kSwOFp157XCSdDvX702...,1770932713050,1242,1770932742218,1770933984645,ARAM,16.3.745.7600,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,...,8126,8140,8135,8014,8009,8100,8100,5001,5008,5008
1,NA1_5487109265,[t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBN...,1770582421604,1723,1770582437192,1770584160231,ARAM,16.3.744.7656,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,...,8226,8210,8237,8014,8009,8200,8200,5001,5008,5008
2,NA1_5478122035,[qJstPPER4-qL0bQLlqUlh7Us_7LSaysoV90xzwhZ9A9Az...,1769697961312,493,1769698090757,1769698583532,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,...,9111,9104,8014,8139,8135,8000,8000,5001,5008,5005
3,NA1_5477493562,[6KWEvXJA31CCUQ77fR-E7QwqyHG_72uEaUTKjnxWHHjFK...,1769638816377,995,1769638836432,1769639830844,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,...,8126,8140,8106,8014,8009,8100,8100,5001,5008,5008
4,NA1_5477456755,[BK4-FEC7sudVe5ysZaGMi8Kkil7rprZ4FacCT_oarNtBz...,1769635418714,1003,1769635456616,1769636459818,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,...,8126,8140,8106,8226,8210,8100,8100,5001,5008,5008


---
# Data Enrichment - ID to Name Mapping 

Raw match data stores, perks, items, and champions as numeric IDs. Use [Community Dragon](https://communitydragon.org/) - an open data source for League assets - to map these IDs to readable names. 

In [11]:
perk = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/perks.json'
perk_styles = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/perkstyles.json'
items = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/items.json'
champ_summary = 'https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/champion-summary.json'

perk_json = requests.get(perk).json()
perk_styles_json = requests.get(perk_styles).json()
items_json = requests.get(items).json()
champ_summary_json = requests.get(champ_summary).json()

In [12]:
def json_extract(obj, key):

    arr = []

    def extract (obj, arr, key):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == key:
                    arr.append(v)
                elif isinstance(v,(dict, list)):
                    extract(v, arr, key)
        elif isinstance(obj, list):
            for item in obj:
                extract (item, arr, key)

        return arr
    
    values = extract(obj, arr, key)
    return values

In [13]:
##Perk dict
perk_ids = json_extract(perk_json,'id')
perk_names = json_extract(perk_json,'name')

perk_dict = {int(i): j for i, j in zip(perk_ids, perk_names)}

perk_columns = [
    'perk_keystone', 'perk_primary_row_1', 'perk_primary_row_2', 'perk_primary_row_3',
    'perk_secondary_row_1', 'perk_secondary_row_2', 'perk_primary_style', 'perk_secondary_style'
]

for col in perk_columns:
    df[col] = df[col].map(perk_dict).fillna(df[col])


##Perk Style Dict
perk_styles_ids = json_extract(perk_styles_json,'id')
perk_styles_names = json_extract(perk_styles_json,'name')

perk_styles_dict = {style['id']: style['name'] for style in perk_styles_json['styles']}

style_columns = ['perk_primary_style', 'perk_secondary_style']

for col in style_columns:
    df[col] = df[col].map(perk_styles_dict).fillna(df[col])

##Shard Dict
shard_dict = {
    5001: 'Health Scaling',
    5002: 'Armor',
    5003: 'Magic Resist',
    5005: 'Attack Speed',
    5007: 'Ability Haste',
    5008: 'Adaptive Force',
    5010: 'Move Speed',
    5011: 'Health',
    5013: 'Tenacity'
}

shard_columns = ['perk_shard_defense', 'perk_shard_flex', 'perk_shard_offense']

for col in shard_columns:
    df[col] = df[col].map(shard_dict).fillna(df[col])

##Items dict
items_dict = {item['id']: item['name'] for item in items_json}

item_columns = ['item0', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6']

for col in item_columns:
    df[col] = df[col].map(items_dict).fillna(df[col])

##Champ Dict
champion_dict = {champ['id']: champ['name'] for champ in champ_summary_json}

df['champion'] = df['champion'].map(champion_dict).fillna(df['champion'])


Preview the enriched DataFrame

In [14]:
pd.set_option('display.max_columns', None)
df.head()

,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,champion,champion_level,kills,deaths,assists,summoner_1_id,summoner_2_id,gold_earned,total_minions_killed,total_neutral_minions_killed,early_surrender,surrender,first_blood,objectives_stolen,objectives_stolen_assists,total_damage_dealt_champions,total_damage_taken,total_damage_shielded_teammates,total_heals_teammates,total_time_crowd_controlled,item0,item1,item2,item3,item4,item5,item6,perk_keystone,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,NA1_5490303087,[oXOnGvsfHrMrBDxweAq6tU7MB5kSwOFp157XCSdDvX702...,1770932713050,1242,1770932742218,1770933984645,ARAM,16.3.745.7600,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Taliyah,18,9,8,24,4,6,13917,50,0,False,False,False,0,0,28017,18248,0,0,360,Luden's Echo,Rabadon's Deathcap,Sorcerer's Shoes,Blackfire Torch,Stormsurge,0,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Treasure Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
1,NA1_5487109265,[t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBN...,1770582421604,1723,1770582437192,1770584160231,ARAM,16.3.744.7656,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Twisted Fate,18,12,11,26,4,6,18975,93,0,False,False,True,0,0,65596,35896,0,0,73,Blackfire Torch,Riftmaker,Ionian Boots of Lucidity,Liandry's Torment,Shadowflame,Rabadon's Deathcap,Poro-Snax,Arcane Comet,Manaflow Band,Transcendence,Scorch,Coup de Grace,Presence of Mind,Sorcery,Sorcery,Health Scaling,Adaptive Force,Adaptive Force
2,NA1_5478122035,[qJstPPER4-qL0bQLlqUlh7Us_7LSaysoV90xzwhZ9A9Az...,1769697961312,493,1769698090757,1769698583532,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Twitch,10,8,5,4,4,6,8274,18,0,False,True,False,0,0,10994,7230,0,0,44,Yun Tal Wildarrows,The Collector,Berserker's Greaves,0,0,0,0,Lethal Tempo,Triumph,Legend: Alacrity,Coup de Grace,Taste of Blood,Treasure Hunter,Precision,Precision,Health Scaling,Adaptive Force,Attack Speed
3,NA1_5477493562,[6KWEvXJA31CCUQ77fR-E7QwqyHG_72uEaUTKjnxWHHjFK...,1769638816377,995,1769638836432,1769639830844,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Malzahar,15,5,10,25,4,3,10787,54,0,False,False,False,0,0,27740,25454,0,0,269,Blackfire Torch,Rylai's Crystal Scepter,Liandry's Torment,Sorcerer's Shoes,Amplifying Tome,0,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
4,NA1_5477456755,[BK4-FEC7sudVe5ysZaGMi8Kkil7rprZ4FacCT_oarNtBz...,1769635418714,1003,1769635456616,1769636459818,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Vel'Koz,16,9,7,15,4,7,12009,34,0,False,False,False,0,0,31226,17458,0,508,197,Malignance,Refillable Potion,Liandry's Torment,Sorcerer's Shoes,Oblivion Orb,Shadowflame,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Manaflow Band,Transcendence,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force


---
# 5. Export to PostgreSQL

Store the enriched match data to a PostgreSQL database using `pangres` — existing records are updated, new ones are added, so no duplicates build up on repeated runs.

Connection credentials are loaded from `.env` via `python-dotenv`.

In [15]:
##Creating connection to postgresql database
import os
from dotenv import load_dotenv
load_dotenv()

from pangres import upsert
from sqlalchemy import text, create_engine

db_username=os.environ.get('db_username')
db_password=os.environ.get('db_password')
db_host=os.environ.get('db_host')
db_port=os.environ.get('db_port')
db_name=os.environ.get('db_name')

def create_db_connection_string(db_username, db_password, db_host, db_port, db_name):
    connection_url = 'postgresql+psycopg2://'+db_username+':'+db_password+'@'+db_host+':'+db_port+'/'+db_name
    return connection_url

conn_url = create_db_connection_string(db_username, db_password, db_host, db_port, db_name)

db_engine = create_engine(conn_url, pool_recycle=3600)

connection = db_engine.connect()

In [16]:
## Create unique primary key from match_id and puuid
df['uuid'] = df['match_id'] + '_' + df['puuid']
df = df.set_index('uuid')

Confirm primary key creation

In [17]:
df.head()

,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,champion,champion_level,kills,deaths,assists,summoner_1_id,summoner_2_id,gold_earned,total_minions_killed,total_neutral_minions_killed,early_surrender,surrender,first_blood,objectives_stolen,objectives_stolen_assists,total_damage_dealt_champions,total_damage_taken,total_damage_shielded_teammates,total_heals_teammates,total_time_crowd_controlled,item0,item1,item2,item3,item4,item5,item6,perk_keystone,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
uuid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
NA1_5490303087_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5490303087,[oXOnGvsfHrMrBDxweAq6tU7MB5kSwOFp157XCSdDvX702...,1770932713050,1242,1770932742218,1770933984645,ARAM,16.3.745.7600,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Taliyah,18,9,8,24,4,6,13917,50,0,False,False,False,0,0,28017,18248,0,0,360,Luden's Echo,Rabadon's Deathcap,Sorcerer's Shoes,Blackfire Torch,Stormsurge,0,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Treasure Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
NA1_5487109265_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5487109265,[t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBN...,1770582421604,1723,1770582437192,1770584160231,ARAM,16.3.744.7656,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Twisted Fate,18,12,11,26,4,6,18975,93,0,False,False,True,0,0,65596,35896,0,0,73,Blackfire Torch,Riftmaker,Ionian Boots of Lucidity,Liandry's Torment,Shadowflame,Rabadon's Deathcap,Poro-Snax,Arcane Comet,Manaflow Band,Transcendence,Scorch,Coup de Grace,Presence of Mind,Sorcery,Sorcery,Health Scaling,Adaptive Force,Adaptive Force
NA1_5478122035_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5478122035,[qJstPPER4-qL0bQLlqUlh7Us_7LSaysoV90xzwhZ9A9Az...,1769697961312,493,1769698090757,1769698583532,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Twitch,10,8,5,4,4,6,8274,18,0,False,True,False,0,0,10994,7230,0,0,44,Yun Tal Wildarrows,The Collector,Berserker's Greaves,0,0,0,0,Lethal Tempo,Triumph,Legend: Alacrity,Coup de Grace,Taste of Blood,Treasure Hunter,Precision,Precision,Health Scaling,Adaptive Force,Attack Speed
NA1_5477493562_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5477493562,[6KWEvXJA31CCUQ77fR-E7QwqyHG_72uEaUTKjnxWHHjFK...,1769638816377,995,1769638836432,1769639830844,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,False,Malzahar,15,5,10,25,4,3,10787,54,0,False,False,False,0,0,27740,25454,0,0,269,Blackfire Torch,Rylai's Crystal Scepter,Liandry's Torment,Sorcerer's Shoes,Amplifying Tome,0,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
NA1_5477456755_t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNmY9UTTO_x1b4Ddt_lhOfHrI4nrSjQFwpw,NA1_5477456755,[BK4-FEC7sudVe5ysZaGMi8Kkil7rprZ4FacCT_oarNtBz...,1769635418714,1003,1769635456616,1769636459818,ARAM,16.2.741.3171,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Vel'Koz,16,9,7,15,4,7,12009,34,0,False,False,False,0,0,31226,17458,0,508,197,Malignance,Refillable Potion,Liandry's Torment,Sorcerer's Shoes,Oblivion Orb,Shadowflame,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Me

In [18]:
# Upsert into PostgreSQL
upsert(con=connection, df=df, schema='soloq', table_name='regional_player_matches', create_table=True, create_schema=True, if_row_exists='update')

In [19]:
connection.commit()

---
# 6. Verify Export - Sample Query
Read back from the database to confirm the data landed correctly. As a quick sanity check, filter for games where the player won.

In [20]:
with db_engine.connect() as connection: 
    df = pd.read_sql(text(f"""select 
                                * 
                            from 
                                soloq.regional_player_matches
                            where
                                win = 'True'"""), connection)

In [21]:
df.head()

,uuid,match_id,participants,game_creation,game_duration,game_start_timestamp,game_end_timestamp,game_mode,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,champion,champion_level,kills,deaths,assists,summoner_1_id,summoner_2_id,gold_earned,total_minions_killed,total_neutral_minions_killed,early_surrender,surrender,first_blood,objectives_stolen,objectives_stolen_assists,total_damage_dealt_champions,total_damage_taken,total_damage_shielded_teammates,total_heals_teammates,total_time_crowd_controlled,item0,item1,item2,item3,item4,item5,item6,perk_keystone,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,NA1_5459748889_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5459748889,[HNuw3XBuOZGaRwAiHLM5rygwtzm85eJLQ9XNmcU2AMB6i...,1767993888497,1515,1767993969586,1767995484671,ARAM,16.1.737.1049,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Ziggs,18,12,8,41,4,6,17491,71,0,False,False,False,0,0,43729,26440,0,0,213,Luden's Echo,Rabadon's Deathcap,Sorcerer's Shoes,Blackfire Torch,Liandry's Torment,Needlessly Large Rod,Poro-Snax,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Manaflow Band,Transcendence,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
1,NA1_5431225481_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5431225481,[IZ99XLyI_SMipD-5FZ2M8YnLB9FAChgvoTkCnuQS4o4ao...,1765140455382,1502,1765140486715,1765141988761,ARAM,15.24.731.5210,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Twisted Fate,18,10,14,27,4,6,16786,56,0,False,False,False,0,0,46081,38670,0,0,53,Blackfire Torch,Rabadon's Deathcap,Liandry's Torment,Sorcerer's Shoes,Shadowflame,Blighting Jewel,Poro-Snax,Arcane Comet,Manaflow Band,Transcendence,Scorch,Coup de Grace,Presence of Mind,Sorcery,Sorcery,Health Scaling,Adaptive Force,Adaptive Force
2,NA1_5430461170_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5430461170,[I6bGeZCz-LAQmaBftneUdzJjNAoDCIAe450i_7npAugs_...,1765073034551,1431,1765073057094,1765074487851,ARAM,15.24.731.5210,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Sion,18,16,15,32,4,32,17649,57,0,False,False,False,0,0,50992,104972,0,0,412,Titanic Hydra,Mercury's Treads,Hollow Radiance,Unending Despair,Thornmail,Heartsteel,Poro-Snax,Grasp of the Undying,Demolish,Conditioning,Overgrowth,Cheap Shot,Ultimate Hunter,Resolve,Resolve,Health Scaling,Adaptive Force,Adaptive Force
3,NA1_5430182151_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5430182151,[VPZuk27xRZRv90IIaA5ijVQhAEu75PSlXdKiBHGgWkYyV...,1765055417637,2143,1765055520039,1765057662614,CLASSIC,15.24.731.5210,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Caitlyn,18,7,4,10,4,21,16063,252,8,False,False,False,0,0,21537,21567,0,0,76,Rapid Firecannon,The Collector,Berserker's Greaves,Yun Tal Wildarrows,Infinity Edge,0,Stealth Ward,Lethal Tempo,Presence of Mind,Legend: Bloodline,Coup de Grace,Absolute Focus,Gathering Storm,Precision,Precision,Health Scaling,Adaptive Force,Attack Speed
4,NA1_5430127649_t6fDViYubCso_U1bup1dVyvndQ-FBtQ...,NA1_5430127649,[t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBN...,1765051438340,1627,1765051476375,1765053103109,ARAM,15.24.731.5210,t6fDViYubCso_U1bup1dVyvndQ-FBtQVNJEsK4Oy4TlBNm...,TaricMidGG,NA1,tRR_9wa2i5X3qPOH1If6vEl6l3FRJhFvDgrc-qSQsSXDYE0,,True,Fiddlesticks,18,10,9,48,4,32,17765,46,0,False,False,False,0,0,52810,43390,0,0,695,Malignance,Shadowflame,Mercury's Treads,Liandry's Torment,Spirit Visage,Blasting Wand,Scarecrow Effigy,Dark Harvest,Cheap Shot,Grisly Mementos,Ultimate Hunter,Coup de Grace,Presence of Mind,Domination,Domination,Health Scaling,Adaptive Force,Adaptive Force
